在工具内部访问运行时信息（如对话历史、用户数据和持久化记忆）

#### 工具内部访问 State（短期记忆）

访问对话历史，跟踪工具调用次数

In [1]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import HumanMessage
from langchain.agents import create_agent, AgentState
from langgraph.types import Command

@tool
def get_last_user_message(runtime: ToolRuntime) -> str:
    """Get the most recent message from the user."""
    messages = runtime.state["messages"]

    # Find the last human message
    for message in reversed(messages):
        if isinstance(message, HumanMessage):
            return message.content

    return "No user messages found"

# Access custom state fields
@tool
def get_user_preference(
    pref_name: str,
    runtime: ToolRuntime
) -> str:
    """Get a user preference value."""
    preferences = runtime.state.get("user_preferences", {})
    return preferences.get(pref_name, "Not set")

class CustomState(AgentState):
    user_preferences: dict
    # user_name: str

agent = create_agent(
    model='deepseek-chat',
    tools=[get_last_user_message, get_user_preference],
    state_schema=CustomState,
)

In [2]:
state = {
    'messages': [HumanMessage('What are my preferences.')],
    'user_preferences': {'style': 'technical', 'verbosity': 'detailed'},
}
state = agent.invoke(state)
state['messages'].append(HumanMessage('What was the last human message?'))
state = agent.invoke(state)
# state['messages'].append(HumanMessage('Set the username to Tom'))
# state = agent.invoke(state)

In [3]:
for msg in state['messages']:
    msg.pretty_print()

================================ Human Message =================================

What are my preferences.
================================== Ai Message ==================================

I'll help you check your preferences. Let me get the most recent user message first to understand the context better, and then I can check for specific preferences.
Tool Calls:
  get_last_user_message (call_00_aMtpCFzSoF4T2TESxumYIzIL)
 Call ID: call_00_aMtpCFzSoF4T2TESxumYIzIL
  Args:
================================= Tool Message =================================
Name: get_last_user_message

What are my preferences.
================================== Ai Message ==================================

Now I can see your question is about checking your preferences. However, to check specific preferences, I need to know which preferences you're interested in. The system allows me to check individual preference values by name, but I don't have a function to list all preferences at once.

Could you tell me 